# NFL Big Data Bowl 2026 — Exploratory Data Analysis

## Goal

The goal of this notebook is to understand the structure of the NFL
player-tracking data and find patterns that can help us build a model
for predicting future player trajectories.

We focus on:

- the structure of `input` and `output`;
- players, plays and frames;
- player movement;
- speed, acceleration and direction;
- ball landing position;
- target players;
- prediction horizon;
- relationships between the input features and future movement.

The final section contains the main findings that can be used during
feature engineering and model development.

## 1. Imports and configuration

For development, the notebook uses the first available week. After the
EDA is finished, `WEEKS_TO_LOAD` can be changed to `None` to load all
available training weeks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/raw")

# Keep this as ["01"] while developing locally.
# Change to None for all available input_2023_wXX.csv files.
WEEKS_TO_LOAD = ["01"]

## 2. Loading the data

In [ ]:
input_files = sorted(DATA_PATH.glob("input_2023_w*.csv"))
output_files = sorted(DATA_PATH.glob("output_2023_w*.csv"))

print(f"Input files found: {len(input_files)}")
print(f"Output files found: {len(output_files)}")

if not input_files:
    raise FileNotFoundError(f"No input files found in {DATA_PATH}")

if WEEKS_TO_LOAD is not None:
    wanted = {f"{int(w):02d}" for w in WEEKS_TO_LOAD}
    input_files = [
        f for f in input_files
        if f.stem[-2:] in wanted
    ]
    output_files = [
        f for f in output_files
        if f.stem[-2:] in wanted
    ]

print("Files used:")
for f in input_files:
    print(" ", f.name)

In [ ]:
input_parts = []
output_parts = []

for file in input_files:
    df = pd.read_csv(file)
    df["week"] = file.stem[-2:]
    input_parts.append(df)

for file in output_files:
    df = pd.read_csv(file)
    df["week"] = file.stem[-2:]
    output_parts.append(df)

input_data = pd.concat(input_parts, ignore_index=True)
output_data = pd.concat(output_parts, ignore_index=True)

print("Input shape:", input_data.shape)
print("Output shape:", output_data.shape)

### First look at the data

In [ ]:
display(input_data.head())
display(output_data.head())

## 3. Dataset structure

The input contains the information available before the prediction
period. The output contains the future `x` and `y` coordinates that the
model has to predict.

In [ ]:
print("INPUT COLUMNS")
for column in input_data.columns:
    print("-", column)

print("\nOUTPUT COLUMNS")
for column in output_data.columns:
    print("-", column)

In [ ]:
summary = pd.DataFrame({
    "dataset": ["input", "output"],
    "rows": [len(input_data), len(output_data)],
    "columns": [input_data.shape[1], output_data.shape[1]],
    "games": [
        input_data["game_id"].nunique(),
        output_data["game_id"].nunique()
    ],
    "plays": [
        input_data[["game_id", "play_id"]].drop_duplicates().shape[0],
        output_data[["game_id", "play_id"]].drop_duplicates().shape[0]
    ],
    "players": [
        input_data["nfl_id"].nunique(),
        output_data["nfl_id"].nunique()
    ]
})

display(summary)

In [ ]:
input_data.info()

In [ ]:
output_data.info()

### Key observation

A play is identified by the combination of `game_id` and `play_id`.
The output has only six main columns: the identifiers, frame number,
and the future `x` and `y` coordinates.

This means that the prediction problem can be viewed as forecasting a
future sequence of `(x, y)` positions for selected players.

## 4. Missing values

In [ ]:
missing = input_data.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No missing values were found in the input data.")
else:
    display(missing.to_frame("missing_values"))

missing_output = output_data.isna().sum().sort_values(ascending=False)
missing_output = missing_output[missing_output > 0]

if missing_output.empty:
    print("No missing values were found in the output data.")
else:
    display(missing_output.to_frame("missing_values"))

### Key observation

Missing values should be checked before modeling. In particular,
dynamic tracking features and static player information should be
handled separately if missing values appear in the full dataset.

## 5. Players and plays

We first examine how many players are present in each play and how many
of them are marked as targets for prediction.

In [ ]:
players_per_play = (
    input_data
    .groupby(["game_id", "play_id"])["nfl_id"]
    .nunique()
)

targets_per_play = (
    input_data[input_data["player_to_predict"]]
    .groupby(["game_id", "play_id"])["nfl_id"]
    .nunique()
)

print("Players per play:")
display(players_per_play.describe())

print("Target players per play:")
display(targets_per_play.describe())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(players_per_play, discrete=True)
plt.title("Number of Players per Play")
plt.xlabel("Number of Players")
plt.ylabel("Number of Plays")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(targets_per_play, discrete=True)
plt.title("Number of Target Players per Play")
plt.xlabel("Number of Target Players")
plt.ylabel("Number of Plays")
plt.show()

### Key observation

The number of players in a play describes the context available to a
trajectory model. The number of target players is smaller than the total
number of players, so the model needs to distinguish target players from
the rest of the players.

## 6. Player roles and prediction targets

In [ ]:
role_target = pd.crosstab(
    input_data["player_role"],
    input_data["player_to_predict"]
)

display(role_target)

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(
    data=input_data,
    x="player_role",
    hue="player_to_predict"
)
plt.title("Player Roles and Prediction Targets")
plt.xlabel("Player Role")
plt.ylabel("Observations")
plt.xticks(rotation=25, ha="right")
plt.show()

In [ ]:
side_target = pd.crosstab(
    input_data["player_side"],
    input_data["player_to_predict"]
)

display(side_target)

### Key observation

`player_role` provides more specific information than `player_side`.
This can be useful because different roles have different movement
patterns and different relationships with the ball landing position.

## 7. Player movement: position, speed and acceleration

In [ ]:
movement_stats = input_data[["x", "y", "s", "a"]].describe().T
display(movement_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=input_data, x="s", bins=50, ax=axes[0])
axes[0].set_title("Distribution of Player Speed")
axes[0].set_xlabel("Speed")
axes[0].set_ylabel("Count")

sns.histplot(data=input_data, x="a", bins=50, ax=axes[1])
axes[1].set_title("Distribution of Player Acceleration")
axes[1].set_xlabel("Acceleration")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
sample_size = min(30000, len(input_data))
movement_sample = input_data.sample(sample_size, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=movement_sample,
    x="s",
    y="a",
    alpha=0.25
)
plt.title("Player Speed vs Acceleration")
plt.xlabel("Speed")
plt.ylabel("Acceleration")
plt.show()

### Key observation

Speed and acceleration describe the current movement state of a player.
They are natural candidates for trajectory forecasting because future
positions depend on the current motion of the player.

## 8. Movement direction and player orientation

`dir` and `o` are angular variables measured in degrees. Because angles
are circular, values close to 0° and 360° represent similar directions.

In [ ]:
display(input_data[["dir", "o"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=input_data, x="dir", bins=60, ax=axes[0])
axes[0].set_title("Movement Direction")
axes[0].set_xlabel("Direction (degrees)")
axes[0].set_ylabel("Count")

sns.histplot(data=input_data, x="o", bins=60, ax=axes[1])
axes[1].set_title("Player Orientation")
axes[1].set_xlabel("Orientation (degrees)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Key observation

Angular features should not necessarily be treated as ordinary linear
numbers. For modeling, a sine/cosine representation can avoid the
artificial discontinuity between 359° and 0°.

## 9. Play direction

In [ ]:
display(input_data["play_direction"].value_counts())

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=input_data, x="play_direction")
plt.title("Play Direction")
plt.xlabel("Direction")
plt.ylabel("Count")
plt.show()

### Key observation

The direction of the play gives information about the orientation of
the action on the field. A useful modeling idea is to normalize the
coordinates relative to the play direction so that similar situations
have a consistent orientation.

## 10. Ball landing position

The input data provides the expected landing point of the ball through
`ball_land_x` and `ball_land_y`. We examine its distribution and its
relationship with player positions.

In [ ]:
display(input_data[["ball_land_x", "ball_land_y"]].describe())

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(
    input_data["ball_land_x"],
    input_data["ball_land_y"],
    alpha=0.08
)
plt.xlabel("Ball landing X")
plt.ylabel("Ball landing Y")
plt.title("Distribution of Ball Landing Positions")
plt.show()

## 11. Distance to the ball landing position

We create a derived feature measuring the Euclidean distance from the
player's current position to the expected ball landing point.

In [ ]:
dx = input_data["ball_land_x"] - input_data["x"]
dy = input_data["ball_land_y"] - input_data["y"]

input_data["distance_to_ball"] = np.sqrt(dx**2 + dy**2)

display(input_data["distance_to_ball"].describe())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=input_data,
    x="distance_to_ball",
    bins=50
)
plt.title("Distance to Ball Landing Position")
plt.xlabel("Distance (yards)")
plt.ylabel("Count")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(
    data=input_data,
    x="player_side",
    y="distance_to_ball",
    ax=axes[0]
)
axes[0].set_title("Distance to Ball Landing by Player Side")
axes[0].set_xlabel("Player Side")
axes[0].set_ylabel("Distance (yards)")

sns.boxplot(
    data=input_data,
    x="player_role",
    y="distance_to_ball",
    ax=axes[1]
)
axes[1].set_title("Distance to Ball Landing by Player Role")
axes[1].set_xlabel("Player Role")
axes[1].set_ylabel("Distance (yards)")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

### Key observation

The distance to the expected ball landing position contains contextual
information about a player's position relative to the pass target.

The distribution also differs between player roles, so `player_role`
together with the distance to the landing position may be useful for
future trajectory prediction.

## 12. Prediction horizon

`num_frames_output` tells us how many future frames have to be predicted.
Because this value is repeated for the rows belonging to one play, we
use one value per `(game_id, play_id)` when analyzing the distribution.

In [ ]:
play_horizon = (
    input_data
    .groupby(["game_id", "play_id"])["num_frames_output"]
    .first()
)

display(play_horizon.describe())

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(play_horizon, discrete=True)
plt.title("Prediction Horizon per Play")
plt.xlabel("Number of Future Frames")
plt.ylabel("Number of Plays")
plt.show()

In [ ]:
prediction_time = play_horizon / 10.0

display(prediction_time.describe())

plt.figure(figsize=(10, 5))
sns.histplot(prediction_time, bins=30)
plt.title("Prediction Horizon in Seconds")
plt.xlabel("Prediction Time (seconds)")
plt.ylabel("Number of Plays")
plt.show()

### Key observation

The prediction horizon determines how far into the future the model must
forecast. It should therefore be explicitly considered when designing
the output layer and the training procedure.

## 13. Example of a single play

A single play is useful for understanding the data visually. We show
the movement of all players and highlight the players whose future
positions are evaluated.

In [ ]:
sample_game = input_data["game_id"].iloc[0]
sample_play = input_data["play_id"].iloc[0]

play_input = input_data[
    (input_data["game_id"] == sample_game) &
    (input_data["play_id"] == sample_play)
].copy()

play_output = output_data[
    (output_data["game_id"] == sample_game) &
    (output_data["play_id"] == sample_play)
].copy()

print("Game ID:", sample_game)
print("Play ID:", sample_play)
print("Input rows:", len(play_input))
print("Input players:", play_input["nfl_id"].nunique())
print("Input frames:", play_input["frame_id"].nunique())
print("Output rows:", len(play_output))
print("Output players:", play_output["nfl_id"].nunique())
print("Output frames:", play_output["frame_id"].nunique())

In [ ]:
plt.figure(figsize=(14, 7))

for nfl_id, player in play_input.groupby("nfl_id"):
    is_target = bool(player["player_to_predict"].iloc[0])

    plt.plot(
        player["x"],
        player["y"],
        linewidth=3 if is_target else 1,
        alpha=1.0 if is_target else 0.18
    )

plt.scatter(
    play_input["ball_land_x"].iloc[0],
    play_input["ball_land_y"].iloc[0],
    marker="*",
    s=300,
    label="Ball landing"
)

plt.xlabel("X position (yards)")
plt.ylabel("Y position (yards)")
plt.title("Player Trajectories and Target Players")
plt.show()

### Key observation

Only selected players are targets for the prediction task. At the same
time, the other players provide important context for understanding the
target player's movement.

## 14. Input trajectory vs future trajectory

The next plot shows the actual future trajectory from `output` together
with the observed trajectory from `input`.

This is the clearest visual representation of the machine-learning task:
the solid part is available to the model, while the dashed part is what
the model needs to predict.

In [ ]:
plt.figure(figsize=(14, 7))

target_ids = set(
    play_input.loc[play_input["player_to_predict"], "nfl_id"]
)

for nfl_id, player in play_input.groupby("nfl_id"):
    is_target = nfl_id in target_ids

    plt.plot(
        player["x"],
        player["y"],
        linewidth=3 if is_target else 1,
        alpha=1.0 if is_target else 0.12
    )

for nfl_id, player in play_output.groupby("nfl_id"):
    plt.plot(
        player["x"],
        player["y"],
        linestyle="--",
        linewidth=2.5
    )

plt.scatter(
    play_input["ball_land_x"].iloc[0],
    play_input["ball_land_y"].iloc[0],
    marker="*",
    s=300,
    label="Ball landing"
)

plt.xlabel("X position (yards)")
plt.ylabel("Y position (yards)")
plt.title("Input and Future Player Trajectories")
plt.show()

### Key observation

Future movement is not simply a straight movement toward the ball
landing point. Players can change direction after the throw.

Therefore, a useful model should combine the player's movement history
with contextual information such as the ball landing position and the
positions and movements of other players.

## 15. Movement during the input sequence

For a target player, it is useful to measure how much the player moves
during the observed history.

In [ ]:
target_input = input_data[input_data["player_to_predict"]].copy()

target_movement = (
    target_input
    .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    .groupby(["game_id", "play_id", "nfl_id"])
    .agg(
        start_x=("x", "first"),
        start_y=("y", "first"),
        end_x=("x", "last"),
        end_y=("y", "last"),
        mean_speed=("s", "mean"),
        max_speed=("s", "max"),
        mean_acceleration=("a", "mean"),
        frames=("frame_id", "count")
    )
    .reset_index()
)

target_movement["observed_displacement"] = np.sqrt(
    (target_movement["end_x"] - target_movement["start_x"]) ** 2 +
    (target_movement["end_y"] - target_movement["start_y"]) ** 2
)

display(target_movement.describe())

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=target_movement.sample(
        min(30000, len(target_movement)),
        random_state=42
    ),
    x="mean_speed",
    y="observed_displacement",
    alpha=0.25
)
plt.title("Mean Speed vs Observed Displacement of Target Players")
plt.xlabel("Mean speed")
plt.ylabel("Observed displacement (yards)")
plt.show()

### Key observation

The observed movement history contains information about how actively a
target player is moving before the prediction period. Features based on
recent speed, acceleration and displacement may therefore help predict
future positions.

## 16. Correlation analysis

Correlation is not enough to determine whether a feature is useful, but
it helps identify basic relationships between numeric variables.

In [ ]:
numeric_columns = [
    "x", "y", "s", "a", "dir", "o",
    "num_frames_output", "ball_land_x",
    "ball_land_y", "distance_to_ball"
]

correlation = input_data[numeric_columns].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    center=0,
    cmap="coolwarm"
)
plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()

### Important note

Correlation should be interpreted carefully here. The data is
sequential, and `dir` and `o` are circular variables. Strong or weak
correlation alone does not establish causal importance for trajectory
prediction.

# 17. Key Findings for Modeling

The EDA suggests several important directions for the future model.

### 1. The task is sequential

Each player is observed over multiple frames before the prediction
period. The model should preserve temporal information rather than using
only one independent row.

### 2. The target is a future trajectory

The output contains future `x` and `y` coordinates. The model therefore
needs to predict a sequence, not just one point.

### 3. The prediction horizon varies

`num_frames_output` determines how many future frames must be predicted.
The model and training procedure should account for different forecast
lengths.

### 4. Player interactions matter

The target player's movement happens in the context of other players.
Relative positions, distances and relative velocities may be useful
features.

### 5. Ball landing position is important

`ball_land_x` and `ball_land_y` provide information about the expected
location of the pass. A derived distance to this point is a natural
candidate feature.

### 6. Player role matters

Different player roles have different movement patterns and different
relationships with the ball landing position.

### 7. Speed and acceleration describe current motion

Recent speed and acceleration can help the model estimate the direction
and magnitude of future movement.

### 8. Direction is circular

`dir` and `o` should be encoded carefully. Sine/cosine transformations
are a possible solution.

### 9. Play direction can be normalized

Transforming the field into a common orientation may make similar plays
more comparable for the model.

### 10. Validation must respect plays

Rows from the same play should not be randomly split between training
and validation. Otherwise, information from the same trajectory can
appear in both sets and produce data leakage.

# Conclusion

The dataset describes player movement as a sequence of observations
before a throw, together with the future trajectory that must be
predicted.

The most promising groups of features for future modeling are:

- recent player positions;
- speed and acceleration;
- movement direction and orientation;
- player role;
- relative positions of other players;
- distance and direction to the ball landing position;
- prediction horizon;
- play direction.

These findings can be used as a starting point for feature engineering
and for choosing a sequence-based or spatio-temporal model.